In [1]:
import sys
sys.path.append("c:/Users/user/Documents/GitHub/AI-INNOVATION-CHALLENGE-2026/backend")

In [2]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

from app.agents.supervisor.state import SupervisorState
from app.agents.supervisor.nodes.supervisor_node import supervisor_node
from app.agents.crm_agent.workflow import build_crm_workflow

c:\Users\user\anaconda3\envs\2026ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-03-02 16:00:02 [info     ] supervisor_initialized        
2026-03-02 16:00:02 [info     ] parser_initialized            
2026-03-02 16:00:02 [info     ] recommender_initialized       
2026-03-02 16:00:02 [info     ] yaml_path_resolved             source=__file__
2026-03-02 16:00:02 [info     ] message_generator_initialized  brand_count=53
2026-03-02 16:00:03 [info     ] quality_checker_initialized   


In [4]:
def should_continue_supervisor_decision(state: SupervisorState):
    if state.get("next_step") == "crm_node":
        return "crm_node"
    elif state.get("next_step") == "search_node":
        return "search_node"
    else:
        return END

In [13]:
def build_marketing_workflow(checkpointer=None):
    workflow = StateGraph(SupervisorState)

    workflow.add_node("supervisor", supervisor_node)
    workflow.add_node("crm_node", build_crm_workflow)

    workflow.add_edge(START, "supervisor")
    workflow.add_conditional_edges(
        "supervisor",
        should_continue_supervisor_decision,
        ["crm_node", END]
    )
    return workflow.compile(checkpointer=checkpointer)

In [14]:
checkpointer = InMemorySaver()

In [15]:
app = build_marketing_workflow(checkpointer)

In [16]:
initial_state: SupervisorState = {
    "messages": [HumanMessage(content="PERSONA_001 에게 보낼 선크림 성능 강조 메시지를 생성해줘")]
}

configurable = {
    "thread_id": "1",
    "session_id": "1",
    "user_id": "son"
}
config = {"configurable": configurable}

In [17]:
response = await app.ainvoke(initial_state, config)

2026-03-02 16:02:30 [info     ] supervisor_decision            next_step=crm_node reason=요청의 최종 목적이 PERSONA_001 대상의 선크림 성능 강조 메시지 생성으로, 페르소나 기반 CRM 메시지 작성을 담당하는 crm_node에 위임합니다.
2026-03-02 16:02:30 [info     ] node_started                  
2026-03-02 16:02:30 [info     ] input_received                 input_length=37
2026-03-02 16:02:30 [info     ] llm_parse_started              operation=llm_parse
2026-03-02 16:02:41 [info     ] llm_parse_completed            duration_ms=11032.0 operation=llm_parse
2026-03-02 16:02:41 [info     ] parse_completed                brands=[] categories=['선블럭'] persona_id=PERSONA_001
2026-03-02 16:02:41 [info     ] node_started                  
2026-03-02 16:02:41 [info     ] request_parsed                 brands=[] categories=['선블럭'] persona_id=PERSONA_001
2026-03-02 16:02:41 [info     ] get_persona_info_started       operation=get_persona_info
2026-03-02 16:02:42 [info     ] persona_fetched                persona_id=PERSONA_001 persona_name=김소현
2026-03-

In [18]:
response

{'step': 0,
 'status': 'running',
 'intermediate': {'request': {'parsed_request': {'persona_id': 'PERSONA_001',
    'purpose': '성분/효능 강조 소개',
    'product_categories': ['선블럭'],
    'brands': [],
    'exclusive_target': None}},
  'recommendation': {'recommended_products': [{'product_id': 'A20251200311',
     'vectordb_id': 'psT5d5sBhJMe5ZsW7D77',
     'product_name': '하이드로 스웻프루프 선크림 SPF50+/PA++++ 50ml',
     'brand': '비레디',
     'product_tag': '선블럭',
     'rating': 4.8,
     'review_count': 60,
     'original_price': 25000,
     'discount_rate': 20,
     'sale_price': 20000,
     'skin_type': ['지성', '복합성'],
     'skin_concerns': ['홍조', '유수분밸런스'],
     'preferred_colors': [],
     'preferred_ingredients': ['히알루론산'],
     'avoided_ingredients': [],
     'preferred_scents': [],
     'values': [],
     'exclusive_product': None,
     'personal_color': [],
     'skin_shades': [],
     'product_image_url': ['https://images-kr.amoremall.com/products/177770000253/177770000253_01.jpg?17459134174

In [20]:
app.get_state(config)

StateSnapshot(values={'step': 0, 'status': 'running', 'intermediate': {}, 'logs': [], 'messages': [HumanMessage(content='PERSONA_001 에게 보낼 선크림 성능 강조 메시지를 생성해줘', additional_kwargs={}, response_metadata={}, id='368de42f-96ab-4965-9b64-dca58d47c89e')], 'input': 'PERSONA_001 에게 보낼 선크림 성능 강조 메시지를 생성해줘'}, next=('crm_node',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11605c-7d95-6a6d-8001-6fa9615a01d9'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}, 'session_id': '1', 'user_id': 'son'}, created_at='2026-03-02T07:02:30.875402+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11605c-5c71-6f62-8000-a2443003df99'}}, tasks=(PregelTask(id='b37497d2-01f7-c4bd-108b-34e3b929bb66', name='crm_node', path=('__pregel_pull', 'crm_node'), error=None, interrupts=(Interrupt(value={'type': 'product_selection', 'recommended_products': [{'product_id': 'A20251200311', 'vectordb_id': 'psT5d5sBhJMe5ZsW7D77', 'product

In [64]:
from pprint import pprint

state = app.get_state(config)

print("\n===== VALUES =====")
pprint(state.values)

print("\n===== NEXT =====")
pprint(state.next)

print("\n===== TASKS =====")
pprint(state.tasks)

print("\n===== INTERRUPTS =====")
pprint(state.interrupts)

print("\n===== METADATA =====")
pprint(state.metadata)


===== VALUES =====
{'input': 'PERSONA_001 에게 보낼 선크림 성능 강조 메시지를 생성해줘',
 'intermediate': {},
 'logs': [],
 'messages': [HumanMessage(content='PERSONA_001 에게 보낼 선크림 성능 강조 메시지를 생성해줘', additional_kwargs={}, response_metadata={}, id='e2cc92e5-1869-4151-ab47-e8c99a68b04e')],
 'status': 'running',
 'step': 0}

===== NEXT =====
('crm_agent',)

===== TASKS =====
(PregelTask(id='cac69bba-2b1a-4256-bc5d-03c3d4e293dd', name='crm_agent', path=('__pregel_pull', 'crm_agent'), error=None, interrupts=(Interrupt(value={'type': 'product_selection', 'recommended_products': [{'product_id': 'A20251200311', 'vectordb_id': 'psT5d5sBhJMe5ZsW7D77', 'product_name': '하이드로 스웻프루프 선크림 SPF50+/PA++++ 50ml', 'brand': '비레디', 'product_tag': '선블럭', 'rating': 4.8, 'review_count': 60, 'original_price': 25000, 'discount_rate': 20, 'sale_price': 20000, 'skin_type': ['지성', '복합성'], 'skin_concerns': ['홍조', '유수분밸런스'], 'preferred_colors': [], 'preferred_ingredients': ['히알루론산'], 'avoided_ingredients': [], 'preferred_scents': [], 'v